# M05 — Human-in-the-loop（人介入）

本 notebook 對應 `README.md`，逐格執行即可。

我們要把「全自動的圖」改成「危險動作前先停下來等人核准」。
三個新工具：`interrupt`（暫停）、`Command(resume=...)`（續跑）、
`get_state_history`（時光旅行）。全都疊在 M04 學的 checkpointer 上。

## 1. 環境準備

載入共用 helper，並取得一個供應商無關的 model。
這格沒有輸出，只是把 `model` 準備好。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
model = get_model()

## 2. 場景：一個「送出 email 前先審核」的圖

狀態裡放三樣東西：
- `topic`：使用者想寄什麼主題的信。
- `draft`：模型寫好的草稿。
- `decision`：人審核後的決定（approve / edit / reject）。
- `result`：最後真的「送出」與否的結果字串。

流程：`write_draft`（寫稿）→ `human_review`（停下來等人）→ `send`（依決定送出或中止）。
這格只定義 State 與工具函式，沒有輸出。

In [ ]:
from typing_extensions import TypedDict
from langchain.messages import SystemMessage, HumanMessage


class EmailState(TypedDict):
    topic: str
    draft: str
    decision: str
    result: str


def draft_email(topic: str) -> str:
    """Ask the model to write a short email draft for the given topic."""
    messages = [
        SystemMessage("你是專業助理，請寫一封簡短、有禮貌的繁體中文 email 草稿，只回信件內容。"),
        HumanMessage(f"主題：{topic}"),
    ]
    return model.invoke(messages).content

## 3. 定義三個 node

重點在 `human_review`：它呼叫 `interrupt(...)`，把草稿拋給人並暫停整張圖。
`interrupt` 的「回傳值」會是人之後用 `Command(resume=...)` 餵回來的東西。

`send` 依 `decision` 決定真的送出、還是中止。這格只定義 node，沒有輸出。

In [ ]:
from langgraph.types import interrupt


def write_draft(state: EmailState) -> dict:
    # Step 1: model writes a draft. This is the "dangerous-looking" content
    # we want a human to approve before it goes out.
    draft = draft_email(state["topic"])
    return {"draft": draft}


def human_review(state: EmailState) -> dict:
    # Step 2: PAUSE here. Hand the draft to a human and wait.
    # The value passed to interrupt() is what the outside world will see.
    # Its return value is whatever the human later resumes with.
    decision = interrupt(
        {
            "question": "這封 email 要送出嗎？(回 'approve' / 'reject' / 或直接給改寫後的內容)",
            "draft": state["draft"],
        }
    )
    return {"decision": decision}


def send(state: EmailState) -> dict:
    # Step 3: act on the human's decision. This is the irreversible action,
    # now gated behind explicit human approval.
    decision = state["decision"]
    if decision == "reject":
        return {"result": "已中止，未送出任何 email。"}
    if decision == "approve":
        return {"result": f"已送出 email：\n{state['draft']}"}
    # Anything else is treated as an edited draft from the human.
    return {"result": f"已送出（人工編輯版）email：\n{decision}"}

## 4. 組圖並 compile（一定要帶 checkpointer）

HITL 完全依賴 checkpointer：暫停時靠它存狀態、續跑時靠它撈狀態。
沒帶 `checkpointer=`，`interrupt` 會直接報錯。
這格印出 ASCII 圖，預期看到 write_draft → human_review → send 的線性結構。

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

builder = StateGraph(EmailState)
builder.add_node("write_draft", write_draft)
builder.add_node("human_review", human_review)
builder.add_node("send", send)
builder.add_edge(START, "write_draft")
builder.add_edge("write_draft", "human_review")
builder.add_edge("human_review", "send")
builder.add_edge("send", END)

# checkpointer is mandatory for human-in-the-loop: it stores the paused state.
graph = builder.compile(checkpointer=InMemorySaver())

print(graph.get_graph().draw_ascii())
# Expected output: a text diagram, roughly
#   __start__ -> write_draft -> human_review -> send -> __end__

## 5. 第一次 invoke：圖會停在 interrupt

第一次 `invoke` 跑到 `human_review` 就暫停並返回。
回傳值裡帶 `__interrupt__`，告訴我們「圖卡住了、在等人」，以及它想給人看的內容。
記得帶 `thread_id`——續跑時要用同一個。

預期輸出：印出草稿，以及一個 `__interrupt__` 物件（裡面有我們在 node 內拋出的 question/draft）。

In [ ]:
config = {"configurable": {"thread_id": "email-1"}}

paused = graph.invoke({"topic": "向客戶說明本週交付延遲一天"}, config)

# The graph did NOT finish — it stopped at human_review.
print("這次回傳含 interrupt 嗎？", "__interrupt__" in paused)
print("待人審核的內容：")
print(paused["__interrupt__"][0].value)

## 6. 模式 A —— 核准：用 Command(resume=...) 續跑

人看完決定核准。我們**不重新 invoke 整張圖**，而是傳 `Command(resume="approve")`，
用同一個 `config`。LangGraph 從 checkpointer 撈回暫停狀態，把 `"approve"`
當成 `interrupt(...)` 的回傳值，讓圖從中斷點接著跑完 `send`。

預期輸出：result 顯示「已送出 email：...」。

In [ ]:
from langgraph.types import Command

approved = graph.invoke(Command(resume="approve"), config)
print(approved["result"])
# Expected output: "已送出 email：" 後面接草稿內容

## 7. 模式 B —— 編輯後再跑

換一條新對話（新的 `thread_id`），這次人不滿意草稿，直接回傳改寫後的內容。
`human_review` 拿到的 `decision` 就是那段文字，`send` 會把它當成人工編輯版送出。

預期輸出：先停在 interrupt，resume 改寫內容後，result 顯示「已送出（人工編輯版）」。

In [ ]:
config_edit = {"configurable": {"thread_id": "email-2"}}

graph.invoke({"topic": "提醒同事明天 10 點開會"}, config_edit)  # pauses at human_review

edited_text = "Hi 各位，提醒一下：明天（週四）上午 10:00 準時開會，請先看過議程。謝謝！"
edited = graph.invoke(Command(resume=edited_text), config_edit)
print(edited["result"])
# Expected output: "已送出（人工編輯版）email：" 後面接 edited_text

## 8. 模式 C —— 拒絕

第三條對話，人決定不送。回 `"reject"`，`send` 直接中止。
這示範同一套 interrupt/resume 機制，只靠 resume 值不同就撐起不同行為。

預期輸出：result 顯示「已中止，未送出任何 email。」

In [ ]:
config_reject = {"configurable": {"thread_id": "email-3"}}
graph.invoke({"topic": "隨手測試一封信"}, config_reject)  # pauses
rejected = graph.invoke(Command(resume="reject"), config_reject)
print(rejected["result"])
# Expected output: "已中止，未送出任何 email。"

## 9. 時光旅行：列出歷史 checkpoint

checkpointer 存的是**每一步**的歷史，不只是最新狀態。
`get_state_history(config)` 回傳一串 `StateSnapshot`（最新在前），
每個含自己的 `config`、`values`、以及 `next`（接下來要跑哪個 node）。

我們用第 6 格那條已完成的對話（thread `email-1`）來看它的歷史。
預期輸出：印出數個 checkpoint，包含 write_draft 之後、human_review 暫停時等時間點。

In [ ]:
history = list(graph.get_state_history(config))  # config = email-1 from cell 5/6
print(f"email-1 共有 {len(history)} 個歷史 checkpoint（最新在前）：\n")
for snap in history:
    checkpoint_id = snap.config["configurable"]["checkpoint_id"]
    next_nodes = snap.next  # tuple of nodes that would run next from here
    print(f"- checkpoint={checkpoint_id[:8]}...  next={next_nodes}  draft有值={bool(snap.values.get('draft'))}")

## 10. 時光旅行：從歷史某點 update_state 後分支重跑

我們倒帶到「`draft` 已寫好、正準備進 `human_review`」的那個 checkpoint，
用 `update_state` 把草稿換成完全不同的內容——這會建立一條**新分支**。
然後從那個分支 `invoke(None, ...)` 續跑（會再停在 human_review），最後核准送出。

重點：同一個起點，因為我們在歷史中途改了狀態，跑出和第 6 格不同的結果。
預期輸出：result 含我們倒帶後塞進去的新草稿。

In [ ]:
# Find the checkpoint whose NEXT step is human_review (i.e. right after draft was written).
target = next(snap for snap in history if snap.next == ("human_review",))

# Rewrite the draft at that past point -> creates a new branch (forked config).
forked_config = graph.update_state(
    target.config,
    {"draft": "（時光旅行分支）這是我們在歷史中途改寫的全新草稿內容。"},
)

# Re-run from the fork. It pauses at human_review again on the new branch.
graph.invoke(None, forked_config)
time_travel_result = graph.invoke(Command(resume="approve"), forked_config)
print(time_travel_result["result"])
# Expected output: "已送出 email：" 後面接「（時光旅行分支）...」那段新草稿

## 🧪 練習 1：加一個「補缺漏資訊」的 interrupt

目前圖直接拿 `topic` 寫稿。請改造成：如果一開始的輸入**沒有收件人**，
就在 `write_draft` 之前加一個 node，用 `interrupt({"ask": "請問收件人是誰?"})`
問人，把人回傳的名字存進 state，再用它客製化稱呼。

提示：
- State 加一個 `recipient: str` 欄位。
- 新 node 內：`name = interrupt({"ask": "請問收件人是誰?"})`，回傳 `{"recipient": name}`。
- 第一次 invoke 會停在這個 interrupt；用 `Command(resume="王經理")` 續跑。
這就是 README 講的「模式 C：補上缺漏資訊」。

## 🧪 練習 2：核准 / 拒絕改用條件邊

目前 `send` 內部用 if 判斷 decision。請改成在 `human_review` 之後用
`add_conditional_edges`，依 decision 路由到 `send_node` 或 `abort_node`，
讓「送出」與「中止」是兩個獨立 node。

提示：
- 寫一個 `route(state) -> str`，回傳 `"send_node"` 或 `"abort_node"`。
- `builder.add_conditional_edges("human_review", route, {...})`（見 M03 的條件路由）。
觀察：把分支從 node 內部「拉到圖結構上」後，流程是不是更一目了然？

## 小結 & 下一步

這個 notebook 把圖從「全自動」升級成「關鍵點交給人」：

- `interrupt(payload)`：在 node 內暫停、把待審內容拋給人。
- 第一次 `invoke` 會停在 interrupt，回傳裡含 `__interrupt__`。
- `Command(resume=value)`：把人的決定餵回去，從中斷點續跑（同一個 `thread_id`）。
- 三種模式（核准/拒絕、編輯、補資訊）共用同一套機制，只差 resume 值。
- `get_state_history` + `update_state`：倒帶到歷史 checkpoint、改狀態、分支重跑。

全部都疊在 M04 的 checkpointer 上——沒有它，暫停與續跑都不成立。

下一站 **M06 — 多智能體系統**：把多個 Agent 編排成一張圖，讓它們分工協作；
而 HITL 正是在這種系統裡安插「人類監督點」的方式。